# Pretraining and Fine-Tuning Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidliwei/pertTF/blob/main/demos/tutorials/pretrain_finetune_tutorial.ipynb)

This notebook pretrains a pertTF model without any perturbation labels, then fine-tunes it with LoRA for perturbation prediction and for cell classification:
1. **Pretrain** a new model with `run_train(pretrain=True)`: masked-gene prediction and reconstruction of each cell's own expression, with every label-based loss off.
2. **Fine-tune perturbation prediction** with `run_lora_pert_train(pert_mode="new")`: new perturbation modules are trained on top of the pretrained model, and predictions are tested on a perturbation left out of both pretraining and fine-tuning.
3. **Fine-tune a classifier** with `run_lora_cls_train`: a second adapter on the same pretrained model.

The demo uses the pancreatic 18-clone dataset and a small model, so it runs in minutes on a GPU. For details on the LoRA options see the LoRA fine-tuning tutorial.

# Install necessary packages

In [ ]:
!pip install scanpy flash-attn huggingface_hub

# Install pertTF

In [ ]:
import sys
# method 1: clone main branch
!git clone https://github.com/davidliwei/pertTF.git
sys.path.insert(0, '/content/pertTF/')

# method 2: pip
#!pip install -i https://test.pypi.org/simple/ pertTF

## 1. Prepare data

pertTF needs an `AnnData` object with log-normalized expression in the layer `X_binned`, highly variable genes in `adata.var['highly_variable']`, and `adata.obs` columns `genotype` (perturbation, `WT` for controls) and `celltype`. Pretraining does not use these labels, but the data loader still reads both columns.

In [2]:
from huggingface_hub import hf_hub_download
import scanpy as sc
import anndata as ad
import numpy as np

hf_hub_download(
    repo_id="weililab/pancreatic_18clone",
    filename="./18clones_seurat.h5ad",
    repo_type="dataset",
    local_dir="./",
)
adata = sc.read_h5ad("./18clones_seurat.h5ad")
adata = ad.AnnData(X=adata.raw.X, obs=adata.obs, var=adata.raw.var)

# Genotype labels from the 'gene' column, e.g. "44_HHEX" -> "HHEX", "46_HHEX_het" -> "HHEXhet", "47_WT" -> "WT"
def clean_genotype(gene_label):
    if gene_label == "NA" or str(gene_label) == "nan":
        return "WT"
    parts = str(gene_label).split("_", 1)
    name = parts[1] if len(parts) > 1 else parts[0]
    return name.replace("_", "").split("/")[0]

adata.obs["genotype"] = adata.obs["gene"].map(clean_genotype)
# Cells without an assigned guide ('NA') are not true controls
adata = adata[adata.obs["gene"] != "NA"].copy()

adata.layers["X_binned"] = adata.X
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
print(adata)

AnnData object with n_obs × n_vars = 18134 × 36601
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sgrna', 'gene', 'nCount_sgRNA', 'nFeature_sgRNA', 'percent.mt', 'RNA_snn_res.0.5', 'seurat_clusters', 'celltype', 'genotype'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg'
    layers: 'X_binned'


## 2. Split the perturbations before pretraining

The split is decided first, because it also decides which cells pretraining may see. Perturbations are split three ways:
- **validation** (`FOXA1`, `OTUD5`): picks the best fine-tuning epoch
- **test** (`HHEX`): predicted at the end as an unseen perturbation
- **training**: all other cells

The cells of the validation and test perturbations are left out of pretraining as well. Even without labels, pretraining on them would show the model the exact expression states it is later asked to predict.

In [3]:
HELD_OUT = ["FOXA1", "OTUD5"]
TEST_PERT = "HHEX"

genotype, celltype = adata.obs["genotype"], adata.obs["celltype"]
is_control = (genotype == "WT").to_numpy()
is_held_out = genotype.isin(HELD_OUT).to_numpy()
is_test = (genotype == TEST_PERT).to_numpy()
# every validation cell type needs both control and perturbed cells
valid_target = is_held_out & celltype.isin(celltype[is_control].unique()).to_numpy()
valid_control = is_control & celltype.isin(celltype[valid_target].unique()).to_numpy()
train_indices = np.where(~is_held_out & ~is_test)[0]
valid_indices = np.where(valid_target | valid_control)[0]

# Pretraining data: control cells and the training perturbations' cells
adata_pretrain = adata[~is_held_out & ~is_test].copy()
print(f"pretraining: {adata_pretrain.n_obs} cells | fine-tuning: train {len(train_indices)}, validation {len(valid_indices)} | test: {is_test.sum()} {TEST_PERT} cells")

pretraining: 13100 cells | fine-tuning: train 13100, validation 5088 | test: 1669 HHEX cells


## 3. Pretrain

`HFPerturbationTFModel.from_adata(adata, config)` builds a new, untrained model: the gene vocabulary and label dictionaries come from the data, the architecture and default training settings from `config`. `run_train` then trains all weights.

With `pretrain=True`, `run_train` overrides any conflicting setting with label-free pretraining: identity objective (`next_cell_pred_type="identity"`, `next_weight=0`), cell type and genotype classifiers off, contrastive loss (`CCE`) off, `perturbation_input` off and lochNESS heads off. What remains is masked-gene prediction and reconstruction of each cell's own expression with the expression decoder (`GEPC`). With `pretrain=False`, the settings alone decide the objective (continued training or full fine-tuning).

The best epoch (lowest validation loss on a random 20% of cells) is restored, and the model is saved with `save_pretrained`, so it loads like any pertTF checkpoint. W&B logging is off unless `wandb_mode="online"` is given. Optimizer and learning-rate schedule settings are described in the guide *Training optimizers and learning-rate schedules*.

In [9]:
from perttf.model.hf import HFPerturbationTFModel

PRETRAIN_DIR = "./pertTF_pretrained"

config = dict(
    # architecture
    layer_size=32,              # embedding size (d_model)
    nlayers=2,                  # transformer layers
    nhead=4,                    # attention heads
    dropout=0.2,
    GEPC=True,                  # expression decoder that reconstructs each cell from its embedding
    distribution="nb",          # expression distribution of that decoder (None for MSE)
    explicit_zero_prob=True,
    fast_transformer=True,
    fast_transformer_backend="flash",
    # input genes: 3000 HVGs plus 1000 other genes sampled per cell
    sampling_mode="hvg",
    n_hvg=3000,
    non_hvg_size=1000,
    log_interval=100,
    # training
    epochs=20,
    batch_size=64,
    lr=1e-3,
    mask_ratio=0.15,
    amp=True,
    seed=42,
)

model = HFPerturbationTFModel.from_adata(adata_pretrain, config)
model.run_train(adata_pretrain, pretrain=True, save_dir=PRETRAIN_DIR)

{'seed': 42, 'batch_size': 64, 'special_tokens': ['<pad>', '<cls>', '<eoc>'], 'n_bins': 51, 'n_hvg': 3000, 'max_seq_len': 4001, 'sampling_mode': 'hvg', 'append_cls': True, 'include_zero_gene': True, 'mask_ratio': 0.15, 'mask_value': -1, 'pad_value': -2, 'pad_token': '<pad>', 'lr': 0.001, 'schedule_ratio': 0.99, 'schedule_interval': 1, 'lr_ADV': 0.001, 'amp': True, 'amp_dtype': 'fp16', 'log_interval': 100, 'layer_size': 32, 'do_train': True, 'GEPC': True, 'CCE': False, 'ADV': False, 'DSBN': False, 'use_batch_label': False, 'perturbation_input': False, 'cell_type_classifier': False, 'genotype_classifier': False, 'cell_type_classifier_weight': 1.0, 'perturbation_classifier_weight': 1.0, 'this_weight': 1.0, 'next_weight': 0, 'next_cell_pred_type': 'identity', 'ecs_thres': 0.0, 'ecs_weight': 1.0, 'dab_weight': 0.0, 'ps_weight': 0.0, 'explicit_zero_prob': True, 'distribution': 'nb', 'use_ot': False, 'dataset_name': 'adata', 'fast_transformer': True, 'non_hvg_size': 1000, 'epochs': 20, 'n_per

HFPerturbationTFModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(36604, 32, padding_idx=0)
    (enc_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.2, inplace=False)
    (linear1): Linear(in_features=1, out_features=32, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=32, out_features=32, bias=True)
    (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x FlashTransformerEncoderLayerVarlen(
        (qkv_proj): Linear(in_features=32, out_features=96, bias=True)
        (out_proj): Linear(in_features=32, out_features=32, bias=True)
        (linear1): Linear(in_features=32, out_features=32, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=32, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_

## 4. Load the pretrained model

The saved directory is a regular pertTF checkpoint. Its training settings record that it was pretrained without labels.

In [5]:
pretrained = HFPerturbationTFModel.from_pretrained(PRETRAIN_DIR, use_fast_transformer=True, fast_transformer_backend="flash")
print({k: pretrained.training_config[k] for k in ["next_cell_pred_type", "genotype_classifier", "cell_type_classifier", "CCE", "distribution"]})

Model loaded. 102 layers transferred.
{'next_cell_pred_type': 'identity', 'genotype_classifier': False, 'cell_type_classifier': False, 'CCE': False, 'distribution': 'nb'}


## 5. Fine-tune perturbation prediction

The pretrained model has no trained perturbation modules, so `pert_mode="new"` builds them: a perturbation encoder, a perturbed-cell encoder and an expression decoder, trained from scratch with LoRA adapters on the transformer. The perturbation dictionary comes from the data.
- `pert_source="esm2"` embeds each perturbed gene from its ESM2 protein embedding, so the held-out validation and test perturbations get an embedding although they are not in training (`"denovo"` would learn one vector per training perturbation only).
- `distribution="nb"` is the expression distribution of the new decoder.

The best epoch is chosen on Pearson delta of the held-out validation perturbations; see the LoRA fine-tuning tutorial for the metric and the other options.

In [10]:
PERT_ADAPTER_DIR = "./lora_pert_adapter"

model = HFPerturbationTFModel.from_pretrained(PRETRAIN_DIR, use_fast_transformer=True, fast_transformer_backend="flash")
peft_pert = model.run_lora_pert_train(
    adata=adata,
    train_indices=train_indices,
    valid_indices=valid_indices,
    epochs=10,
    batch_size=128,
    lr=1e-3,
    lora_config=model.build_lora_config(r=32, lora_alpha=32, lora_dropout=0.1),
    save_dir=PERT_ADAPTER_DIR,
    seed=42,
    pert_mode="new",
    pert_source="esm2",
    distribution="nb",
)

Model loaded. 102 layers transferred.


/autofs/projects-t3/lilab/yangqisu/repos/pertf_benchmark/./external_worktrees/perttf_lora_pert_modes/perttf/utils/split_check.py:183: UserWarning: Validation control groups with fewer than 30 cells: ('EnP', 19)
  _warn_groups(f"Validation control groups with fewer than {min_cells} cells", low_control_groups)
/autofs/projects-t3/lilab/yangqisu/repos/pertf_benchmark/./external_worktrees/perttf_lora_pert_modes/perttf/utils/split_check.py:185: UserWarning: Validation perturbation-context groups with fewer than 30 cells: ('ESC', 'OTUD5', 22), ('EnP', 'OTUD5', 9), ('Endothelial', 'OTUD5', 28)
  _warn_groups(


Train/validation split check: {'mode': 'pert', 'n_train': 13100, 'n_valid': 5088, 'n_overlapping_controls': 1723, 'combination_type_counts': {'seen_combination': 0, 'context_transfer': 0, 'unseen_perturbation': 14, 'unseen_context': 0, 'unseen_context_and_perturbation': 0}, 'context_exposure_counts': {'seen_combination': 0, 'fewshot': 7, 'control_only': 0, 'unseen': 0}}
Initializing PertTFUniDataManager: Creating vocab and mappings...
sampling_mode is hvg, sampling 3000 HVGs + 1000 non-HVGs for training
Initialization complete.


/autofs/projects-t3/lilab/yangqisu/.local/miniforge3/envs/torch_flashv3/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


pertTF - INFO - | epoch   1 | 100/103 batches | lr 0.00100000 | ms/batch 62.00 | loss  0.87 | mse  0.05 | mre 33830.13 |mse_next  0.05 | mre_next 34162.13 |cls  0.00 | pert  0.00 | ps  0.00 | ps_next  0.00 |gepc  0.32 |gepc_next  0.39 |
Epoch 1/10 | val pearson_delta: 0.1027 | mse_delta: 0.0195 | mvc_next: 0.2505 | groups: 10
pertTF - INFO - | epoch   2 | 100/103 batches | lr 0.00099000 | ms/batch 60.32 | loss  0.66 | mse  0.05 | mre 33099.33 |mse_next  0.05 | mre_next 33451.98 |cls  0.00 | pert  0.00 | ps  0.00 | ps_next  0.00 |gepc  0.24 |gepc_next  0.26 |
Epoch 2/10 | val pearson_delta: 0.1045 | mse_delta: 0.0186 | mvc_next: 0.2473 | groups: 10
pertTF - INFO - | epoch   3 | 100/103 batches | lr 0.00098010 | ms/batch 60.29 | loss  0.65 | mse  0.05 | mre 33102.65 |mse_next  0.05 | mre_next 33465.93 |cls  0.00 | pert  0.00 | ps  0.00 | ps_next  0.00 |gepc  0.23 |gepc_next  0.25 |
Epoch 3/10 | val pearson_delta: 0.1046 | mse_delta: 0.0189 | mvc_next: 0.2466 | groups: 10
pertTF - INFO - 

### Predict the unseen test perturbation

The adapter is reloaded on the pretrained checkpoint (`load_lora_adapter` rebuilds the new modules with their perturbation dictionary and distribution, and checks that the base is the checkpoint the adapter was trained on). Control cells of the cell types that have `HHEX` cells are each asked to predict `HHEX`, and the predictions are compared with the real `HHEX` cells: per (cell type, perturbation) group with at least 30 cells, Pearson delta is the correlation over all genes between the predicted and the observed change from control.

In [11]:
import pandas as pd
from perttf.model.train_function import eval_testdata
from perttf.utils.pert_metrics import compute_metrics_from_anndata, prediction_scale

peft_pert = HFPerturbationTFModel.load_lora_adapter(
    PRETRAIN_DIR, PERT_ADAPTER_DIR, use_fast_transformer=True, fast_transformer_backend="flash"
)
peft_pert.to("cuda")
peft_pert.eval()
base = peft_pert.get_base_model()

query_control = is_control & celltype.isin(celltype[is_test].unique()).to_numpy()
adata_query = adata[query_control].copy()
adata_query.obs["genotype_next"] = TEST_PERT
adata_real = adata[is_test | query_control].copy()   # observed test cells and the same controls

adata_pred = eval_testdata(
    peft_pert,
    adata_query,
    None,
    train_data_dict={"genotype_to_index": base.genotype_to_index, "vocab": base.vocab, "cell_type_to_index": base.cell_type_to_index},
    config=base.training_config,
    mvc_full_expr=True,
    predict_expr=True,
    sample=True,          # draw expression from the predicted distribution
    sizefactor=True,      # at each control cell's library size
    sample_seed=42,
)
metrics = compute_metrics_from_anndata(adata_real, adata_pred, real_layer="X_binned", prediction_scale=prediction_scale(base.distribution))
display(pd.DataFrame(metrics["per_group"]).T[["pearson_delta", "mse_delta", "ttest_de_overlap_at_n", "ttest_de_direction_match", "n_real_cells", "n_predicted_cells"]])
display(pd.Series(metrics["aggregate"], name=TEST_PERT))

Model loaded. 102 layers transferred.
pertTF - INFO - 36601 genes shared between model vocab's 36604 and anndata's 36601 genes


,pearson_delta,mse_delta,ttest_de_overlap_at_n,ttest_de_direction_match,n_real_cells,n_predicted_cells
DE|HHEX,0.018852,0.013214,0.123762,0.527228,936.0,731.0
ESC|HHEX,0.091360,0.016401,0.000000,0.500000,63.0,86.0
Endothelial|HHEX,-0.070515,0.025406,0.000000,0.666667,192.0,59.0
Liver|HHEX,0.017995,0.014957,0.000000,0.555556,244.0,291.0
PP|HHEX,0.201462,0.019653,0.088523,0.394236,31.0,464.0
Stromal|HHEX,0.072614,0.021788,0.000000,0.875000,201.0,73.0


pearson_delta                 0.055295
mse_delta                     0.018570
ttest_de_overlap_at_n         0.035381
ttest_de_direction_match      0.586448
n_real_degs                 398.333333
n_evaluated_groups            6.000000
Name: HHEX, dtype: float64

## 6. Fine-tune a classifier on the same pretrained model

One pretrained model can serve several adapters. `run_lora_cls_train` trains a new head on the cell embedding together with LoRA adapters; here it predicts the cell type. `predict_cls` embeds cells with the adapted model and applies the head.

In [12]:
from sklearn.model_selection import train_test_split
import pandas as pd
CLS_ADAPTER_DIR = "./lora_celltype_adapter"

model = HFPerturbationTFModel.from_pretrained(PRETRAIN_DIR, use_fast_transformer=True, fast_transformer_backend="flash")
cls_train, cls_valid = train_test_split(np.arange(adata.n_obs), test_size=0.2, stratify=adata.obs["celltype"], random_state=42)
peft_cls = model.run_lora_cls_train(
    adata=adata,
    cls_col="celltype",
    cls_task="classification",
    train_indices=cls_train,
    valid_indices=cls_valid,
    epochs=5,
    batch_size=128,
    lr=1e-3,
    save_dir=CLS_ADAPTER_DIR,
    seed=42,
)

adata_valid = adata[cls_valid].copy()
adata_valid.obs["predicted_celltype_ft"] = peft_cls.predict_cls(adata_valid)
accuracy = (adata_valid.obs["predicted_celltype_ft"] == adata_valid.obs["celltype"]).mean()
print(f"validation accuracy: {accuracy:.3f}")
pd.crosstab(adata_valid.obs["celltype"], adata_valid.obs["predicted_celltype_ft"])

Model loaded. 102 layers transferred.
[lora] model genotype_to_index missing labels from current adata; rebuilding genotype mapping from adata.
Train/validation split check: {'mode': 'identity', 'n_train': 14507, 'n_valid': 3627}
Initializing PertTFUniDataManager: Creating vocab and mappings...
sampling_mode is hvg, sampling 3000 HVGs + 1000 non-HVGs for training
Initialization complete.
Epoch 1/5 | train head: 0.6941 recon: 0.3943 | val head: 0.3182 recon: 0.3915
Epoch 2/5 | train head: 0.3226 recon: 0.3939 | val head: 0.2088 recon: 0.3910
Epoch 3/5 | train head: 0.2533 recon: 0.3929 | val head: 0.1724 recon: 0.3908
Epoch 4/5 | train head: 0.2276 recon: 0.3937 | val head: 0.1941 recon: 0.3901
Epoch 5/5 | train head: 0.2154 recon: 0.3930 | val head: 0.1520 recon: 0.3893
Restored best model from epoch 5
Saved PEFT adapter to: /autofs/projects-t3/lilab/yangqisu/repos/pertf_benchmark/lora_celltype_adapter
pertTF - INFO - 36601 genes shared between model vocab's 36604 and anndata's 36601 g

predicted_celltype_ft,DE,ESC,EnP,Endothelial,Liver,PP,Stromal
celltype,,,,,,,
DE,1434,35,0,0,10,13,8
ESC,54,262,0,0,3,0,1
EnP,0,0,85,2,2,1,0
Endothelial,0,0,1,120,0,0,4
Liver,0,0,0,1,918,27,3
PP,0,0,2,0,32,354,0
Stromal,1,0,0,1,3,0,250


## 7. Other options

- **Full fine-tuning instead of LoRA**: `run_train` with `pretrain=False` trains all weights with the objective given by the settings, e.g. continue from a checkpoint trained for perturbation prediction:
  ```python
  model = HFPerturbationTFModel.from_pretrained("weililab/perttf-tiny-perturb-5k-nb")
  model.run_train(adata, train_config={"epochs": 5, "lr": 1e-4, "perturbation_checkpoint_metric": "pearson_delta_sample"},
                  train_indices=train_indices, valid_indices=valid_indices, save_dir="./full_finetune")
  ```
- **Perturbation embeddings**: `pert_source` also takes `"genept"`, `"gears"` or combinations such as `"esm2+genept"`.
- **Expression distribution of the new decoder**: `distribution` takes `"nb"`, `"zinb"`, `"hnb"`, `"zig"`, `"pois"`, `"zipois"` or `None` (MSE).
- **Fine-tuning a checkpoint that was already trained for perturbation prediction**: `pert_mode="ft"`, see the LoRA fine-tuning tutorial.